# Tahap 2b J4 (J3 + augmentasi ganti latar): JENIS tanaman dengan TRANSFER LEARNING MobileNetV2 (9 kelas) di Google Colab

**Sebelum mulai**
1. Lokal: `python jenis.py` lalu `python scripts/make_colab_bundle.py --task jenis` → upload `jenis_bundle.zip` ke **Google Drive → MyDrive/**.
2. Colab: **Runtime → Change runtime type → T4 GPU**, lalu **Runtime → Disconnect and delete runtime**.
3. **Runtime → Run all** (sel TEST otomatis dilewati). Pastikan sel 1 menampilkan GPU.
4. Kirim output sel 5 (VAL) ke Claude. Aturan stop (CLAUDE.md "Tahap 2"): J4: dipakai jika macro F1 val >= 0,8132 DAN test >= 0,80 DAN 12 foto user > 6/12 (CLAUDE.md J4).
   Setelah disetujui: `RUN_TEST = True` di sel 7, jalankan sel 7 & 8.

## 1. Setup

In [ ]:
!nvidia-smi -L
!pip -q install tensorflow==2.20.0 keras==3.10.0 ai-edge-litert
!python -c "import tensorflow as tf, keras; print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU', tf.config.list_physical_devices('GPU'))"

## 2. Data dari Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/proj && mkdir -p /content/proj
!cp /content/drive/MyDrive/jenis_bundle.zip /content/ && unzip -q -o /content/jenis_bundle.zip -d /content/proj
%cd /content/proj

import csv, glob
from collections import Counter
rows = list(csv.DictReader(open('data/jenis/split.csv', encoding='utf-8')))
n_jpg = len(glob.glob('data/jenis/*/*/*.jpg'))
assert n_jpg == len(rows), f'jumlah citra {n_jpg} != split.csv {len(rows)}'
print('citra:', n_jpg)  # lihat output jenis.py
print('train per kelas:', dict(sorted(Counter(r['label'] for r in rows if r['split'] == 'train').items())))

## 3. Cek arsitektur (9 kelas)

In [ ]:
!python -c "import jenis; from model import build_transfer; m, _ = build_transfer(len(jenis.CLASSES)); print(m.output_shape, m.count_params(), 'parameter')"

## 4. Training J4 (SATU run): fase 1 kepala (base beku) lalu fase 2 fine-tune 40 lapisan atas. Bobot ImageNet diunduh otomatis.

In [ ]:
%cd /content/proj
!rm -rf models && python train.py --task jenis --arch mobilenetv2 --epochs 15 --finetune-epochs 20 --patience 6 2>&1 | grep -E 'args:|GPU:|Found|val_accuracy|Restoring|early stopping|Fine-tune|Model terbaik'
!python evaluate.py --task jenis --split val > /dev/null 2>&1 && rm -rf models_J4 && cp -r models models_J4
# simpan LANGSUNG ke Drive: runtime Colab bisa ter-reset saat idle
!rm -rf /content/drive/MyDrive/models_J4 && cp -r models_J4 /content/drive/MyDrive/
print('selesai, tersimpan juga di Drive: MyDrive/models_J4')

## 5. Ringkasan VAL (kirim output sel ini ke Claude)

In [ ]:
%cd /content
import os
from IPython.display import Image, display
if not os.path.isdir('/content/proj'):  # pulihkan bila runtime ter-reset
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/proj && cp /content/drive/MyDrive/jenis_bundle.zip /content/ && unzip -q -o /content/jenis_bundle.zip -d /content/proj
%cd /content/proj
if not os.path.isdir('models_J4'):
    !cp -r /content/drive/MyDrive/models_J4 .
assert os.path.isdir('models_J4'), 'models_J4 tidak ada di runtime maupun Drive: jalankan ulang sel 2 & 4'
!sed -n '/== KESELURUHAN/,/weighted avg/p' models_J4/eval_val/report.txt | grep -vE '^  (true|pred)'
display(Image('models_J4/history.png'))
display(Image('models_J4/eval_val/confusion_matrix.png'))

## 6. Ekspor TFLite (models/model_jenis.tflite)

In [ ]:
%cd /content/proj
!rm -rf models && cp -r models_J4 models && python inference.py --export --task jenis 2>&1 | tail -1

## 7. Evaluasi TEST: SEKALI saja, setelah VAL disetujui

In [ ]:
%cd /content/proj
from IPython.display import Image, display
RUN_TEST = False  # ubah ke True setelah hasil VAL disetujui

if RUN_TEST:
    !python evaluate.py --task jenis --split test > /dev/null 2>&1
    !sed -n '/== KESELURUHAN/,/weighted avg/p' models/eval_test/report.txt | grep -vE '^  (true|pred)'
    display(Image('models/eval_test/confusion_matrix.png'))
else:
    print('Dilewati: set RUN_TEST = True setelah VAL disetujui.')

## 8. Simpan ke Drive + unduh `models_jenis_j4.zip`

In [ ]:
%cd /content/proj
!cd models && zip -q -r /content/models_jenis_j4.zip .
!cp /content/models_jenis_j4.zip /content/drive/MyDrive/
!unzip -l /content/models_jenis_j4.zip | grep -E 'best.keras|model_jenis.tflite|report.txt'
from google.colab import files
files.download('/content/models_jenis_j4.zip')